In [43]:
import pandas as pd
import numpy as np


In [44]:
table_life = {
    "Bearing_Life" : [1000,1100,1200,1300,1400,1500,1600,1700,1800,1900],
    "Probability" : [0.10,0.14,0.24,0.14,0.12,0.10,0.06,0.05,0.03,0.02]
}
bearing_life = pd.DataFrame(table_life)
bearing_life

,Bearing_Life,Probability
0,1000,0.10
1,1100,0.14
2,1200,0.24
3,1300,0.14
4,1400,0.12
5,1500,0.10
6,1600,0.06
7,1700,0.05
8,1800,0.03
9,1900,0.02


In [45]:
bearing_life["cumsum"] = bearing_life["Probability"].cumsum()
bearing_life

,Bearing_Life,Probability,cumsum
0,1000,0.10,0.10
1,1100,0.14,0.24
2,1200,0.24,0.48
3,1300,0.14,0.62
4,1400,0.12,0.74
5,1500,0.10,0.84
6,1600,0.06,0.90
7,1700,0.05,0.95
8,1800,0.03,0.98
9,1900,0.02,1.00


In [46]:
bearing_life['upper'] = (bearing_life['cumsum']*100).astype(int)
bearing_life

,Bearing_Life,Probability,cumsum,upper
0,1000,0.10,0.10,10
1,1100,0.14,0.24,24
2,1200,0.24,0.48,48
3,1300,0.14,0.62,62
4,1400,0.12,0.74,74
5,1500,0.10,0.84,84
6,1600,0.06,0.90,89
7,1700,0.05,0.95,95
8,1800,0.03,0.98,98
9,1900,0.02,1.00,100


In [47]:
bearing_life["lower"] = (bearing_life["upper"].shift(1).fillna(0)+1).astype(int)
bearing_life

,Bearing_Life,Probability,cumsum,upper,lower
0,1000,0.10,0.10,10,1
1,1100,0.14,0.24,24,11
2,1200,0.24,0.48,48,25
3,1300,0.14,0.62,62,49
4,1400,0.12,0.74,74,63
5,1500,0.10,0.84,84,75
6,1600,0.06,0.90,89,85
7,1700,0.05,0.95,95,90
8,1800,0.03,0.98,98,96
9,1900,0.02,1.00,100,99


In [48]:
bearing_life.iloc[-1,3] = 0 
bearing_life

,Bearing_Life,Probability,cumsum,upper,lower
0,1000,0.10,0.10,10,1
1,1100,0.14,0.24,24,11
2,1200,0.24,0.48,48,25
3,1300,0.14,0.62,62,49
4,1400,0.12,0.74,74,63
5,1500,0.10,0.84,84,75
6,1600,0.06,0.90,89,85
7,1700,0.05,0.95,95,90
8,1800,0.03,0.98,98,96
9,1900,0.02,1.00,0,99


In [49]:
delay_data = {
    "delay_time(min)":[4,6,8],
    "probability":[0.3,0.6,0.1]
}

delay  = pd.DataFrame(delay_data)
delay["cumsum"] = delay['probability'].cumsum()
delay['upper'] =(delay['cumsum']*10)
delay['lower'] = delay['upper'].shift(1).fillna(0)+1 
delay.iloc[-1,3] = 0 

delay

,delay_time(min),probability,cumsum,upper,lower
0,4,0.3,0.3,3.0,1.0
1,6,0.6,0.9,9.0,4.0
2,8,0.1,1.0,0.0,10.0


In [50]:
def get_bearing_life_val(random_digit,df): 
    for idx,row in df.iterrows():
        low = row['lower']
        high = row['upper']
        if low<= random_digit <= high:
            return row["Bearing_Life"]
        elif high<= random_digit <= low:
            return row["Bearing_Life"]
        
get_bearing_life_val(1,bearing_life)

np.float64(1000.0)

In [51]:
def get_delay_val(random_digit,df): 
    for idx,row in df.iterrows():
        low = row['lower']
        high = row['upper']
        if low<= random_digit <= high:
            return row["delay_time(min)"]
        elif high<= random_digit <= low:
            return row["delay_time(min)"]
        
get_delay_val(0,delay)


np.float64(8.0)

In [52]:
np.random.randint(0,10,1)

array([0], dtype=int32)

In [53]:
def four_brearing_together_policy(target_hours):
    ans = []

    for i in range(4):
        total_time = 0
        while True:
            random_number_bearing = np.random.randint(0,100,1).item() 
            random_number_delay = np.random.randint(0,10,1).item() 
            life_time = get_bearing_life_val(random_number_bearing,df=bearing_life)
            delay_time = get_delay_val(random_number_delay,df=delay)
            total_time += life_time
            ans.append(
                {   "Bearing_ID":(i+1),
                    "RD_Bearing":random_number_bearing,
                    "Life_Time":life_time,
                    "Cum_Life_Time": total_time,
                    "RD_Delay":random_number_delay,
                    "Delay_time":delay_time
                }
            )
        
            if total_time>=target_hours:
                break
    return pd.DataFrame(ans)

four_bearing  = four_brearing_together_policy(18000)
four_bearing.sample(5)

,Bearing_ID,RD_Bearing,Life_Time,Cum_Life_Time,RD_Delay,Delay_time
9,1,5,1000.0,13000.0,1,4.0
15,2,48,1200.0,3100.0,4,6.0
28,3,42,1200.0,1200.0,6,6.0
13,1,92,1700.0,18100.0,8,6.0
6,1,4,1000.0,9400.0,9,6.0


In [54]:
four_bearing[four_bearing["Bearing_ID"]==1].reset_index()

,index,Bearing_ID,RD_Bearing,Life_Time,Cum_Life_Time,RD_Delay,Delay_time
0,0,1,16,1100.0,1100.0,1,4.0
1,1,1,57,1300.0,2400.0,8,6.0
2,2,1,92,1700.0,4100.0,0,8.0
3,3,1,79,1500.0,5600.0,1,4.0
4,4,1,76,1500.0,7100.0,4,6.0
5,5,1,52,1300.0,8400.0,3,4.0
6,6,1,4,1000.0,9400.0,9,6.0
7,7,1,59,1300.0,10700.0,9,6.0
8,8,1,59,1300.0,12000.0,9,6.0
9,9,1,5,1000.0,13000.0,1,4.0


In [55]:
four_bearing[four_bearing["Bearing_ID"]==2].reset_index()

,index,Bearing_ID,RD_Bearing,Life_Time,Cum_Life_Time,RD_Delay,Delay_time
0,14,2,99,1900.0,1900.0,4,6.0
1,15,2,48,1200.0,3100.0,4,6.0
2,16,2,43,1200.0,4300.0,8,6.0
3,17,2,20,1100.0,5400.0,6,6.0
4,18,2,18,1100.0,6500.0,4,6.0
5,19,2,71,1400.0,7900.0,8,6.0
6,20,2,64,1400.0,9300.0,5,6.0
7,21,2,97,1800.0,11100.0,1,4.0
8,22,2,81,1500.0,12600.0,1,4.0
9,23,2,15,1100.0,13700.0,5,6.0


In [56]:
four_bearing[four_bearing["Bearing_ID"]==3].reset_index()

,index,Bearing_ID,RD_Bearing,Life_Time,Cum_Life_Time,RD_Delay,Delay_time
0,28,3,42,1200.0,1200.0,6,6.0
1,29,3,85,1600.0,2800.0,5,6.0
2,30,3,74,1400.0,4200.0,6,6.0
3,31,3,57,1300.0,5500.0,0,8.0
4,32,3,20,1100.0,6600.0,0,8.0
5,33,3,34,1200.0,7800.0,7,6.0
6,34,3,42,1200.0,9000.0,7,6.0
7,35,3,26,1200.0,10200.0,9,6.0
8,36,3,82,1500.0,11700.0,7,6.0
9,37,3,80,1500.0,13200.0,9,6.0


In [57]:
four_bearing[four_bearing["Bearing_ID"]==4].reset_index()

,index,Bearing_ID,RD_Bearing,Life_Time,Cum_Life_Time,RD_Delay,Delay_time
0,42,4,33,1200.0,1200.0,6,6.0
1,43,4,12,1100.0,2300.0,4,6.0
2,44,4,75,1500.0,3800.0,1,4.0
3,45,4,90,1700.0,5500.0,9,6.0
4,46,4,33,1200.0,6700.0,7,6.0
5,47,4,13,1100.0,7800.0,1,4.0
6,48,4,94,1700.0,9500.0,1,4.0
7,49,4,73,1400.0,10900.0,4,6.0
8,50,4,73,1400.0,12300.0,9,6.0
9,51,4,30,1200.0,13500.0,2,4.0


In [58]:
total_bearing = len(four_bearing)
total_delay  = four_bearing["Delay_time"].sum().item()

bearnig_cost = total_bearing * 20 
print("Bearing Cost: ",bearnig_cost)


repaire_cost = (total_delay/60) * 25 
print("Repaire_cost: ",repaire_cost)

downltime_cost = (total_delay + (total_bearing*20))* 5 
print("Downtime Cost: ",downltime_cost)


total_cost_when_change_bearing_four_sep = (bearnig_cost+repaire_cost+downltime_cost)

print("Total cost", total_cost_when_change_bearing_four_sep)


Bearing Cost:  1120
Repaire_cost:  131.66666666666666
Downtime Cost:  7180.0
Total cost 8431.666666666666


In [59]:
np.random.randint(0,10,1).item()

9

In [60]:
def get_value_table(bearing_idx,val_idx):
    try:
        val = four_bearing[four_bearing["Bearing_ID"]==bearing_idx]["Life_Time"].iloc[val_idx]
        return val 
    except:
        random_number = np.random.randint(0,10,1).item() 
        return get_bearing_life_val(random_number,df=bearing_life)
    

def chage_four_bearing_at_once(target_hours):
    cnt = 0 
    all_recoards = []
    total_time = 0 
    while True: 
        
        bearing1 = get_value_table(1,cnt)
        bearing2 = get_value_table(2,cnt)
        bearing3 = get_value_table(3,cnt)
        bearing4 = get_value_table(4,cnt)
        cnt+=1
        
        first_failure = min(bearing1,bearing2,bearing3,bearing4)
        total_time += first_failure
        
        rd = np.random.randint(0,10,1).item()
        del_time = get_delay_val(rd,df=delay)
        
        all_recoards.append({
            "Bearing1":bearing1,
            "Bearing2":bearing2,
            "Bearing3":bearing3,
            "Bearing4":bearing4,
            "First_Failure":first_failure,
            "cumsum_lifetime": total_time,
            "RD": rd,
            "delay": del_time 
        })
        
        if total_time>=target_hours:
            break 
    return pd.DataFrame(all_recoards)

change_all = chage_four_bearing_at_once(18000)
change_all

,Bearing1,Bearing2,Bearing3,Bearing4,First_Failure,cumsum_lifetime,RD,delay
0,1100.0,1900.0,1200.0,1200.0,1100.0,1100.0,9,6.0
1,1300.0,1200.0,1600.0,1100.0,1100.0,2200.0,6,6.0
2,1700.0,1200.0,1400.0,1500.0,1200.0,3400.0,0,8.0
3,1500.0,1100.0,1300.0,1700.0,1100.0,4500.0,7,6.0
4,1500.0,1100.0,1100.0,1200.0,1100.0,5600.0,0,8.0
5,1300.0,1400.0,1200.0,1100.0,1100.0,6700.0,1,4.0
6,1000.0,1400.0,1200.0,1700.0,1000.0,7700.0,2,4.0
7,1300.0,1800.0,1200.0,1400.0,1200.0,8900.0,8,6.0
8,1300.0,1500.0,1500.0,1400.0,1300.0,10200.0,1,4.0
9,1000.0,1100.0,1500.0,1200.0,1000.0,11200.0,2,4.0


In [61]:
total_bearing = len(change_all)
total_delay  = change_all["delay"].sum().item()

bearnig_cost = total_bearing * 20 
print("Bearing Cost: ",bearnig_cost)

repaire_cost = (total_delay/60) * 25 
print("Repaire_cost: ",repaire_cost)
 
downltime_cost = (total_delay + (total_bearing*20))* 5 
print("Downtime Cost: ",downltime_cost)

total_cost_replace_4_at_once = (bearnig_cost+repaire_cost+downltime_cost)

print("Total cost", total_cost_replace_4_at_once)

Bearing Cost:  340
Repaire_cost:  39.166666666666664
Downtime Cost:  2170.0
Total cost 2549.1666666666665


In [62]:
print("repalce one by one : ",total_cost_when_change_bearing_four_sep)
print("repalce_4_at_once : ",total_cost_replace_4_at_once)

repalce one by one :  8431.666666666666
repalce_4_at_once :  2549.1666666666665


# 3 Bearing vs 4 Bearing Comparison


In [63]:
def generate_bearing_policy(num_bearings, target_hours):
    records = []

    for bearing_id in range(1, num_bearings + 1):
        total_time = 0
        while True:
            random_number_bearing = np.random.randint(1, 101, 1).item()
            random_number_delay = np.random.randint(1, 11, 1).item()
            life_time = get_bearing_life_val(random_number_bearing, df=bearing_life)
            delay_time = get_delay_val(random_number_delay, df=delay)
            total_time += life_time
            records.append(
                {
                    "Bearing_ID": bearing_id,
                    "RD_Bearing": random_number_bearing,
                    "Life_Time": life_time,
                    "Cum_Life_Time": total_time,
                    "RD_Delay": random_number_delay,
                    "Delay_time": delay_time,
                }
            )

            if total_time >= target_hours:
                break

    return pd.DataFrame(records)


def get_value_table(table_df, bearing_idx, val_idx):
    bearing_values = table_df[table_df["Bearing_ID"] == bearing_idx]["Life_Time"]
    if val_idx < len(bearing_values):
        return bearing_values.iloc[val_idx]

    random_number = np.random.randint(1, 101, 1).item()
    return get_bearing_life_val(random_number, df=bearing_life)


def change_bearing_at_once(num_bearings, life_table, target_hours):
    cnt = 0
    all_records = []
    total_time = 0

    while True:
        bearing_values = [get_value_table(life_table, bearing_idx, cnt) for bearing_idx in range(1, num_bearings + 1)]
        cnt += 1

        first_failure = min(bearing_values)
        total_time += first_failure

        rd = np.random.randint(1, 11, 1).item()
        del_time = get_delay_val(rd, df=delay)

        record = {f"Bearing{idx}": bearing_values[idx - 1] for idx in range(1, num_bearings + 1)}
        record.update(
            {
                "First_Failure": first_failure,
                "cumsum_lifetime": total_time,
                "RD": rd,
                "delay": del_time,
            }
        )
        all_records.append(record)

        if total_time >= target_hours:
            break

    return pd.DataFrame(all_records)


def calculate_total_cost(df, delay_col):
    total_bearing = len(df)
    total_delay = df[delay_col].sum().item()

    bearing_cost = total_bearing * 20
    repair_cost = (total_delay / 60) * 25
    downtime_cost = (total_delay + (total_bearing * 20)) * 5
    total_cost = bearing_cost + repair_cost + downtime_cost

    return {
        "total_bearing": total_bearing,
        "total_delay": total_delay,
        "bearing_cost": bearing_cost,
        "repair_cost": repair_cost,
        "downtime_cost": downtime_cost,
        "total_cost": total_cost,
    }


three_bearing = generate_bearing_policy(3, 18000)
three_change_all = change_bearing_at_once(3, three_bearing, 18000)

four_bearing_new = generate_bearing_policy(4, 18000)
four_change_all_new = change_bearing_at_once(4, four_bearing_new, 18000)

three_separate_cost = calculate_total_cost(three_bearing, "Delay_time")
three_at_once_cost = calculate_total_cost(three_change_all, "delay")
four_separate_cost = calculate_total_cost(four_bearing_new, "Delay_time")
four_at_once_cost = calculate_total_cost(four_change_all_new, "delay")

comparison = pd.DataFrame(
    [
        {
            "System": "3 Bearing - Replace one by one",
            "Total Cost": three_separate_cost["total_cost"],
        },
        {
            "System": "3 Bearing - Replace all at once",
            "Total Cost": three_at_once_cost["total_cost"],
        },
        {
            "System": "4 Bearing - Replace one by one",
            "Total Cost": four_separate_cost["total_cost"],
        },
        {
            "System": "4 Bearing - Replace all at once",
            "Total Cost": four_at_once_cost["total_cost"],
        },
    ]
)

print("3 Bearing - Replace one by one:", three_separate_cost["total_cost"])
print("3 Bearing - Replace all at once:", three_at_once_cost["total_cost"])
print("4 Bearing - Replace one by one:", four_separate_cost["total_cost"])
print("4 Bearing - Replace all at once:", four_at_once_cost["total_cost"])

comparison

TypeError: unsupported operand type(s) for +: 'float' and 'NoneType'